In [4]:
import tensorflow as tf
import numpy as np
import cv2
from tensorflow.keras.models import load_model

In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

In [6]:
img_size = 128
batch_size = 16
epochs = 20

In [7]:
# augmentation
train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 15,
    zoom_range = 0.15,
    brightness_range = [0.6, 1.4],
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    channel_shift_range=30.0
)

In [8]:
val_datagen = ImageDataGenerator(
    rescale = 1./255
)

In [9]:
train_generator = train_datagen.flow_from_directory(
    "../dataset_sign_detection_3.0/train",
    target_size = (img_size, img_size),
    batch_size = batch_size,
    class_mode = 'categorical',
    shuffle=True
)

Found 10000 images belonging to 10 classes.


In [10]:
val_generator = val_datagen.flow_from_directory(
    "../dataset_sign_detection_3.0/val",
    target_size = (img_size, img_size),
    batch_size = batch_size,
    class_mode = 'categorical',
    shuffle=False
)

Found 2000 images belonging to 10 classes.


In [11]:
num_classes = train_generator.num_classes
print("Class indices: ", train_generator.class_indices)
print("Number of classes: ", num_classes)

Class indices:  {'approve': 0, 'call_me': 1, 'disapprove': 2, 'fist': 3, 'fuck_off': 4, 'loser': 5, 'ok': 6, 'peace': 7, 'rock': 8, 'stop': 9}
Number of classes:  10


In [12]:
model = Sequential([

    # Block 1
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(img_size, img_size, 3)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # Block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.3),

    # Block 4
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.3),

    # Block 5
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.3),

    Flatten(),

    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),

    Dense(256, activation='relu'),
    Dropout(0.4),

    Dense(num_classes, activation='softmax')
])

In [13]:
model.compile(
    optimizer = Adam(learning_rate=0.0001),
    loss = 'categorical_crossentropy',
    metrics=['accuracy']
)

In [12]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 128, 128, 32)      896       
                                                                 
 batch_normalization (BatchN  (None, 128, 128, 32)     128       
 ormalization)                                                   
                                                                 
 conv2d_1 (Conv2D)           (None, 128, 128, 32)      9248      
                                                                 
 batch_normalization_1 (Batc  (None, 128, 128, 32)     128       
 hNormalization)                                                 
                                                                 
 max_pooling2d (MaxPooling2D  (None, 64, 64, 32)       0         
 )                                                               
                                                        

In [13]:
history = model.fit(
    train_generator,
    validation_data = val_generator,
    epochs=epochs,
    verbose=1
)

Epoch 1/20
625/625 [==============================] - 141s 210ms/step - loss: 2.9156 - accuracy: 0.1698 - val_loss: 3.0054 - val_accuracy: 0.1420
Epoch 2/20
625/625 [==============================] - 61s 97ms/step - loss: 2.0065 - accuracy: 0.3402 - val_loss: 1.6703 - val_accuracy: 0.3890
Epoch 3/20
625/625 [==============================] - 59s 94ms/step - loss: 1.3972 - accuracy: 0.5216 - val_loss: 0.9834 - val_accuracy: 0.6560
Epoch 4/20
625/625 [==============================] - 59s 94ms/step - loss: 0.9555 - accuracy: 0.6659 - val_loss: 0.4582 - val_accuracy: 0.8295
Epoch 5/20
625/625 [==============================] - 55s 88ms/step - loss: 0.6744 - accuracy: 0.7635 - val_loss: 0.4850 - val_accuracy: 0.7950
Epoch 6/20
625/625 [==============================] - 55s 87ms/step - loss: 0.5006 - accuracy: 0.8294 - val_loss: 0.8815 - val_accuracy: 0.7005
Epoch 7/20
625/625 [==============================] - 52s 84ms/step - loss: 0.4111 - accuracy: 0.8625 - val_loss: 0.1372 - val_accurac

In [ ]:
#model.save("gesture_model_3.keras")

In [41]:
model = load_model("gesture_model_3.keras")
#img_size = 128 approve  call_me    
class_names = ['approve', 'call_me', 'disapprove', 'fist', 'one', 'loser', 'ok', 'peace', 'rock', 'stop']

In [15]:
def predict_frame(frame):
  img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  img = cv2.resize(img, (img_size, img_size))
  img = img / 255.0
  img = np.expand_dims(img, axis = 0)

  predictions = model.predict(img, verbose=0)
  confidence = np.max(predictions)
  class_index = np.argmax(predictions)
  gesture = class_names[class_index]

  return gesture, confidence

In [16]:
last_prediction = ""
stable_count = 0

In [ ]:
cap = cv2.VideoCapture(0)

import pyttsx3

engine = pyttsx3.init()
engine.setProperty('rate', 150)  

last_spoken = ""  

while True:
  ret, frame = cap.read()
  if not ret:
     break
  frame = cv2.flip(frame, 1)

  h, w, _ = frame.shape

  # ROI
  box_size = 250
  margin = 20
  x1 = w - box_size - margin
  y1 = margin
  x2 = w - margin
  y2 = margin + box_size

  cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

  roi = frame[y1:y2, x1:x2]

  gesture, confidence = predict_frame(roi)

  if confidence > 0.6:
    if gesture == last_prediction:
        stable_count += 1
    else:
        stable_count = 1
        last_prediction = gesture
    
    if stable_count > 10:
        print("Confirmed: ", gesture)
        if gesture != last_spoken:          
          engine.say(gesture.replace('_', ' '))
          engine.runAndWait()
          last_spoken = gesture
        stable_count = 0

  if confidence > 0.6:
    cv2.putText(frame, f"{gesture} ({confidence:.2f})",
                (x1, y1+300),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0,255,0), 2)
    
  cv2.imshow("Gesture Recognition", frame)
  if cv2.waitKey(1) & 0xFF == 27:
    break

cap.release()
cv2.destroyAllWindows()

Confirmed:  teri_behen_chowd_doongaa_kilometer_k_hisaab_say
Confirmed:  maa_chudale_bhosdike
Confirmed:  maa_chudale_bhosdike
Confirmed:  maa_chudale_bhosdike
Confirmed:  maa_chudale_bhosdike
Confirmed:  maa_chudale_bhosdike
Confirmed:  teri_behen_chowd_doongaa_kilometer_k_hisaab_say
Confirmed:  teri_behen_chowd_doongaa_kilometer_k_hisaab_say
Confirmed:  teri_behen_chowd_doongaa_kilometer_k_hisaab_say
Confirmed:  teri_behen_chowd_doongaa_kilometer_k_hisaab_say
Confirmed:  teri_behen_chowd_doongaa_kilometer_k_hisaab_say
Confirmed:  teri_behen_chowd_doongaa_kilometer_k_hisaab_say
Confirmed:  teri_behen_chowd_doongaa_kilometer_k_hisaab_say
Confirmed:  teri_behen_chowd_doongaa_kilometer_k_hisaab_say
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confirmed:  stop
Confir